# 🚀 ADA — Adaptive Depth Architecture: Colab TPU Pretraining

This notebook trains **ADA-Nano** on Google Colab's **TPU v2-8** runtime.

**What it does:**
- Streams real data from [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu-score-2) (no download needed)
- Detects TPU/GPU/CPU automatically
- Trains with cosine LR schedule, gradient clipping, and adaptive depth loss
- Saves checkpoints to Google Drive

> **Runtime Setup:** Go to `Runtime → Change runtime type → TPU v2-8`

In [ ]:
# ============================================================
# Cell 1: Environment Detection & TPU Setup
# ============================================================
import os
import sys
import time

# Detect runtime environment
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
DEVICE_TYPE = 'cpu'
HAS_XLA = False

if IN_COLAB:
    print('[INFO] Running in Google Colab')
else:
    print('[INFO] Running in local / non-Colab environment')

# Check for TPU
try:
    import torch
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    device = xm.xla_device()
    DEVICE_TYPE = 'tpu'
    HAS_XLA = True
    n_cores = xm.xrt_world_size()
    print(f'[OK] TPU Detected: {device} | Cores: {n_cores}')
except Exception as e:
    import torch
    if torch.cuda.is_available():
        DEVICE_TYPE = 'cuda'
        device = torch.device('cuda')
        print(f'[OK] GPU Detected: {torch.cuda.get_device_name(0)}')
    else:
        DEVICE_TYPE = 'cpu'
        device = torch.device('cpu')
        print(f'[WARN] No accelerator found, using CPU. XLA error: {e}')

print(f'\n>> Accelerator: {DEVICE_TYPE.upper()}')
print(f'>> PyTorch:     {torch.__version__}')

In [ ]:
# ============================================================
# Cell 2: Install ADA Package + Dependencies
# ============================================================
%%capture install_log
!pip install -q einops omegaconf datasets tiktoken tokenizers pyyaml
!pip install -q git+https://github.com/sec24me094-commits/ada-tpu-experiments.git

# Verify installation
import ada
from ada.config import ADAConfig
from ada.model import ADAModel
print(f'[OK] ADA package installed successfully')
print(f'[OK] ADAConfig fields: {len(ADAConfig.__dataclass_fields__)} hyperparameters')

In [ ]:
# ============================================================
# Cell 3: Mount Google Drive for Checkpoints
# ============================================================
SAVE_DIR = '/content/ada_checkpoints'

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        SAVE_DIR = '/content/drive/MyDrive/ada_checkpoints'
        print(f'[OK] Google Drive mounted. Checkpoints will save to: {SAVE_DIR}')
    except Exception as e:
        print(f'[WARN] Drive mount failed ({e}). Saving to /content/ (ephemeral).')
        SAVE_DIR = '/content/ada_checkpoints'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'>> Checkpoint directory: {SAVE_DIR}')

In [ ]:
# ============================================================
# Cell 4: Configure ADA-Nano Architecture
# ============================================================
from ada.config import ADAConfig
from ada.model import ADAModel

# Batch size tuning per accelerator
BATCH_SIZES = {'tpu': 32, 'cuda': 16, 'cpu': 4}
SEQ_LENS    = {'tpu': 512, 'cuda': 512, 'cpu': 128}

config = ADAConfig(
    name='ada-nano-colab-tpu',
    vocab_size=32000,
    hidden_dim=768,
    num_layers=12,
    max_seq_len=SEQ_LENS[DEVICE_TYPE],
    micro_batch_size=BATCH_SIZES[DEVICE_TYPE],
    # Mamba-2 SSD
    use_ssm=True,
    ssm_state_dim=128,
    ssm_num_heads=8,
    ssm_head_dim=96,
    ssm_conv_kernel=4,
    # MoE
    use_moe=True,
    moe_num_experts=8,
    moe_top_k=2,
    moe_expert_hidden_mult=2.0,
    lambda_balance=0.01,
    # Sparse GQA
    use_dynamic_sparse_attn=True,
    num_query_heads=12,
    num_kv_heads=4,
    attn_sparsity_target=0.5,
    # Adaptive Depth
    use_adaptive_depth=True,
    depth_budget_tau=0.75,
    lambda_depth=0.01,
    # Training
    learning_rate=3e-4,
    min_learning_rate=3e-5,
    weight_decay=0.1,
    warmup_steps=200,
)

print('=== ADA-Nano Configuration ===')
print(f'  Layers:       {config.num_layers}')
print(f'  Hidden Dim:   {config.hidden_dim}')
print(f'  Seq Length:   {config.max_seq_len}')
print(f'  Batch Size:   {config.micro_batch_size}')
print(f'  Experts:      {config.moe_num_experts} (top-{config.moe_top_k})')
print(f'  Depth Budget: {config.depth_budget_tau}')
print(f'  Device:       {DEVICE_TYPE.upper()}')

In [ ]:
# ============================================================
# Cell 5: Load FineWeb-Edu Dataset (Streaming)
# ============================================================
from datasets import load_dataset
import torch

print('Loading FineWeb-Edu dataset (streaming mode, no disk download)...')
dataset = load_dataset(
    'HuggingFaceFW/fineweb-edu-score-2',
    split='train',
    streaming=True,
)

# Simple tokenizer: byte-level fallback for fast iteration
# (Replace with trained BPE for production runs)
def tokenize_batch(texts, seq_len, vocab_size):
    """Fast byte-level tokenization for pretraining smoke test."""
    batch_ids = []
    for text in texts:
        ids = [b % vocab_size for b in text.encode('utf-8')[:seq_len]]
        # Pad or truncate
        if len(ids) < seq_len:
            ids = ids + [0] * (seq_len - len(ids))
        batch_ids.append(ids[:seq_len])
    return torch.tensor(batch_ids, dtype=torch.long)

# Create data iterator
data_iter = iter(dataset)

def get_batch():
    """Fetch next batch from streaming dataset."""
    texts = []
    for _ in range(config.micro_batch_size):
        try:
            sample = next(data_iter)
            texts.append(sample['text'])
        except StopIteration:
            # Wrap around (shouldn't happen with 1.3T token dataset)
            texts.append('The quick brown fox jumps over the lazy dog.')
    return tokenize_batch(texts, config.max_seq_len, config.vocab_size)

# Smoke test
test_batch = get_batch()
print(f'[OK] Data pipeline ready: batch shape = {test_batch.shape}')
print(f'     Sample token range: [{test_batch.min().item()}, {test_batch.max().item()}]')

In [ ]:
# ============================================================
# Cell 6: Build Model, Optimizer & Scheduler
# ============================================================
from ada.model import ADAModel
from ada.training.optimizer import build_optimizer, build_lr_scheduler
from ada.training.loss import compute_loss

# Training config
MAX_STEPS   = 500
LOG_EVERY   = 25
SAVE_EVERY  = 100
GRAD_ACCUM  = 4 if DEVICE_TYPE == 'tpu' else 2  # Effective batch = micro_batch * accum

# Target device
if DEVICE_TYPE == 'tpu':
    import torch_xla.core.xla_model as xm
    target_device = xm.xla_device()
elif DEVICE_TYPE == 'cuda':
    target_device = torch.device('cuda')
else:
    target_device = torch.device('cpu')

# Build model
model = ADAModel(config).to(target_device)

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[OK] Model built on {DEVICE_TYPE.upper()}')
print(f'     Total params:     {total_params:,}')
print(f'     Trainable params: {trainable_params:,}')
print(f'     Memory estimate:  ~{total_params * 4 / 1e9:.2f} GB (fp32)')

# Build optimizer & scheduler
optimizer = build_optimizer(model, config)
scheduler = build_lr_scheduler(optimizer, config, total_steps=MAX_STEPS)

print(f'\n[OK] Optimizer: AdamW (lr={config.learning_rate}, wd={config.weight_decay})')
print(f'[OK] Scheduler: Cosine warmup ({config.warmup_steps} warmup steps)')
print(f'[OK] Gradient accumulation: {GRAD_ACCUM} steps')
print(f'     Effective batch: {config.micro_batch_size * GRAD_ACCUM} sequences')
print(f'     Effective tokens/step: {config.micro_batch_size * GRAD_ACCUM * config.max_seq_len:,}')

In [ ]:
# ============================================================
# Cell 7: Training Loop
# ============================================================
import json

print(f'\n{"="*60}')
print(f'  ADA-Nano Pretraining on {DEVICE_TYPE.upper()}')
print(f'  Steps: {MAX_STEPS} | Log: every {LOG_EVERY} | Save: every {SAVE_EVERY}')
print(f'{"="*60}\n')

model.train()
training_log = []
best_loss = float('inf')
total_tokens = 0
start_time = time.time()

for step in range(1, MAX_STEPS + 1):
    step_loss = 0.0
    step_lm = 0.0
    step_depth = 0.0
    step_bal = 0.0

    # --- Gradient Accumulation ---
    for micro_step in range(GRAD_ACCUM):
        input_ids = get_batch().to(target_device)
        labels = input_ids.clone()

        output = model(input_ids, inference=False)
        loss_out = compute_loss(output, labels, config)
        loss = loss_out.total / GRAD_ACCUM
        loss.backward()

        step_loss  += loss_out.total.item() / GRAD_ACCUM
        step_lm    += loss_out.lm_loss.item() / GRAD_ACCUM
        step_depth += loss_out.depth_loss.item() / GRAD_ACCUM
        step_bal   += loss_out.balance_loss.item() / GRAD_ACCUM

        total_tokens += config.micro_batch_size * config.max_seq_len

    # --- Optimizer Step ---
    if DEVICE_TYPE == 'tpu':
        xm.reduce_gradients(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        xm.optimizer_step(optimizer)
        xm.mark_step()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()

    scheduler.step()
    optimizer.zero_grad(set_to_none=True)

    # --- Logging ---
    if step % LOG_EVERY == 0 or step == 1 or step == MAX_STEPS:
        elapsed = time.time() - start_time
        tok_s = total_tokens / max(elapsed, 1e-5)
        lr_now = scheduler.get_last_lr()[0]

        log_entry = {
            'step': step,
            'loss': round(step_loss, 4),
            'lm_loss': round(step_lm, 4),
            'depth_loss': round(step_depth, 4),
            'balance_loss': round(step_bal, 4),
            'lr': lr_now,
            'tok_s': int(tok_s),
            'elapsed_s': round(elapsed, 1),
            'total_tokens': total_tokens,
        }
        training_log.append(log_entry)

        print(
            f'[Step {step:4d}/{MAX_STEPS}] '
            f'Loss: {step_loss:.4f} '
            f'(LM: {step_lm:.4f} Depth: {step_depth:.4f} Bal: {step_bal:.4f}) | '
            f'Throughput: {tok_s:,.0f} tok/s | '
            f'LR: {lr_now:.2e} | '
            f'Tokens: {total_tokens:,}'
        )

    # --- Checkpointing ---
    if step % SAVE_EVERY == 0 or step == MAX_STEPS:
        ckpt = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config.__dict__,
            'loss': step_loss,
            'total_tokens': total_tokens,
        }
        ckpt_path = os.path.join(SAVE_DIR, f'ada_nano_step_{step}.pt')
        if DEVICE_TYPE == 'tpu':
            import torch_xla.utils.serialization as xser
            xser.save(ckpt, ckpt_path)
        else:
            torch.save(ckpt, ckpt_path)
        print(f'  >> Checkpoint saved: {ckpt_path}')

        if step_loss < best_loss:
            best_loss = step_loss
            best_path = os.path.join(SAVE_DIR, 'ada_nano_best.pt')
            if DEVICE_TYPE == 'tpu':
                xser.save(ckpt, best_path)
            else:
                torch.save(ckpt, best_path)
            print(f'  >> New best model saved: {best_path}')

# Save training log
log_path = os.path.join(SAVE_DIR, 'training_log.json')
with open(log_path, 'w') as f:
    json.dump(training_log, f, indent=2)

elapsed_total = time.time() - start_time
print(f'\n{"="*60}')
print(f'  Training Complete!')
print(f'  Total time:   {elapsed_total/60:.1f} min')
print(f'  Total tokens: {total_tokens:,}')
print(f'  Final loss:   {step_loss:.4f}')
print(f'  Best loss:    {best_loss:.4f}')
print(f'  Checkpoints:  {SAVE_DIR}')
print(f'{"="*60}')

In [ ]:
# ============================================================
# Cell 8: Training Loss Visualization
# ============================================================
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    steps  = [e['step'] for e in training_log]
    losses = [e['loss'] for e in training_log]
    lm_ls  = [e['lm_loss'] for e in training_log]
    depth_ls = [e['depth_loss'] for e in training_log]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Total loss
    ax1.plot(steps, losses, 'b-', linewidth=2, label='Total Loss')
    ax1.plot(steps, lm_ls, 'r--', linewidth=1.5, label='LM Loss')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('ADA-Nano Training Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Throughput
    tok_s = [e['tok_s'] for e in training_log]
    ax2.plot(steps, tok_s, 'g-', linewidth=2)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Tokens/sec')
    ax2.set_title(f'Throughput ({DEVICE_TYPE.upper()})')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(SAVE_DIR, 'training_curves.png')
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f'[OK] Training curves saved: {plot_path}')
except ImportError:
    print('[WARN] matplotlib not available, skipping plots')

In [ ]:
# ============================================================
# Cell 9: Routing Analytics & Depth Verification
# ============================================================
print('\nEvaluating Adaptive Depth Exit Routing...\n')
model.eval()

eval_input = get_batch().to(target_device)

with torch.no_grad():
    out = model(eval_input, inference=True, return_routing_stats=True)

if DEVICE_TYPE == 'tpu':
    xm.mark_step()  # Materialize XLA tensors

print('--- Routing Analytics ---')
print(f'Mean Exit Depth:      {out.depth_stats.get("mean_depth", "N/A"):.2f} / {config.num_layers} layers')
print(f'Expert Entropy:       {out.depth_stats.get("expert_entropy", "N/A"):.4f}')
print(f'Sparse Mask Density:  {out.depth_stats.get("sparse_mask_density", "N/A")}')

if 'per_token_depth' in out.depth_stats:
    depths = out.depth_stats['per_token_depth'].float()
    print(f'\nDepth Distribution:')
    print(f'  Min:    {depths.min().item():.0f}')
    print(f'  Max:    {depths.max().item():.0f}')
    print(f'  Median: {depths.median().item():.0f}')
    print(f'  Std:    {depths.std().item():.2f}')

print('\n[OK] Verification complete. Dynamic depth and expert sparsity verified!')

In [ ]:
# ============================================================
# Cell 10: Summary & Next Steps
# ============================================================
print('\n' + '='*60)
print('  ADA-Nano Pretraining Summary')
print('='*60)
print(f'  Accelerator:    {DEVICE_TYPE.upper()}')
print(f'  Model Params:   {trainable_params:,}')
print(f'  Steps:          {MAX_STEPS}')
print(f'  Tokens Trained: {total_tokens:,}')
print(f'  Final Loss:     {training_log[-1]["loss"]:.4f}')
print(f'  Best Loss:      {best_loss:.4f}')
print(f'  Throughput:     {training_log[-1]["tok_s"]:,} tok/s')
print(f'  Checkpoints:    {SAVE_DIR}')
print('='*60)
print()
print('Next steps:')
print('  1. Increase MAX_STEPS to 5000+ for meaningful convergence')
print('  2. Swap byte-level tokenizer for trained BPE (ada/tokenizer.py)')
print('  3. Run ablation studies (disable MoE, depth, sparse attn)')
print('  4. Export to HuggingFace format (scripts/export_to_hf.py)')